# Title

This program:

1. Loads csv.
2. Separates attributes and labels.
3. Splits dataset.
4. Imports random forest model.
5. Trains the model.
6. Plots train vs validation behavior (Accuracy).
7. Makes predictions for test.
8. Calculates metrics and show them on classification report.
9. Plots confusion Matrix.


## Imports

In [24]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns

import sys
import os

## 1. Load dataset

In [43]:
# Load embeddings
df_train_emb = pd.read_csv("Embeddings/ai_detection/train_embeddings.csv", sep=",")
df_val_emb = pd.read_csv("Embeddings/ai_detection/val_embeddings.csv", sep=",")
df_test_emb = pd.read_csv("Embeddings/ai_detection/test_embeddings.csv", sep=",")

df_train_emb.head()

,0,1,2,3,4,5,6,7,8,9,...,759,760,761,762,763,764,765,766,767,label
0,0.806076,-0.471976,-0.288829,0.895243,1.448630,0.930668,0.179704,-1.247050,0.347692,-1.050877,...,1.158921,0.361953,0.600954,-1.154133,0.640420,1.370007,0.238205,0.120366,-0.545586,0
1,0.826584,-0.464866,-0.265668,0.863336,1.520433,0.938907,0.197947,-1.299503,0.406856,-1.002167,...,1.215423,0.341694,0.563445,-1.141032,0.648643,1.420650,0.248102,0.189239,-0.547863,0
2,-0.242146,0.442891,0.000355,-0.532203,-1.637065,-2.307497,0.133571,1.696639,0.085779,0.645553,...,-0.907270,-0.232473,0.170094,0.795657,0.019162,-1.447572,-0.519326,-0.275860,0.249933,1
3,-0.294113,0.431339,-0.047076,-0.448314,-1.679870,-2.285092,0.170854,1.707610,0.108657,0.603445,...,-0.855614,-0.415144,0.221501,0.741392,0.020038,-1.325863,-0.457287,-0.256800,0.401724,1
4,-0.260345,0.448323,-0.037299,-0.491892,-1.665105,-2.298147,0.132397,1.697994,0.165164,0.650408,...,-0.901356,-0.316128,0.119760,0.846162,0.043563,-1.380406,-0.513420,-0.193510,0.289815,1


In [39]:
# Load stylometric + AST features
df_train_feat = pd.read_csv("../Dataframes/df_ai/df_ai_train.csv", sep=",")
df_val_feat = pd.read_csv("../Dataframes/df_ai/df_ai_val.csv", sep=",")
df_test_feat = pd.read_csv("../Dataframes/df_ai/df_ai_test.csv", sep=",")

df_train_feat.head()

,code,label,approx_tokens,comment_density,avg_line_length,line_length_variance,blank_line_ratio,num_classes,num_methods,num_if,num_for,num_while,num_switch,o_complexity,max_depth,total_nodes,num_literals,num_ids,unique_ids,id_diversity
0,private Set<Integer> perNodeRelease(final C th...,0,101,0.0,61.882353,1268.339100,0.055556,0.0,0.0,0.0,0.0,0.0,0.0,1.0,6.0,0.0,0.0,63.0,30.0,0.476190
1,@Override\r\n public AuthenticationStatus f...,0,23,0.0,34.777778,702.395062,0.100000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,19.0,15.0,0.789474
2,public void callWorkListenerWithError(WorkCont...,1,28,0.0,49.500000,2072.250000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,5.0,0.0,0.0,7.0,7.0,1.000000
3,public void setSubscription(Subscription s) {\...,1,14,0.0,33.250000,500.187500,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,6.0,5.0,0.833333
4,public boolean getDialogContentInset(int theme...,1,20,0.0,44.666667,1461.222222,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,21.0,16.0,0.761905


In [41]:
EXTRA_FEATURE_COLUMNS = [
    "comment_density",
    "avg_line_length",
    "line_length_variance",
    "blank_line_ratio",
    "num_classes",
    "num_methods",
    "num_if",
    "num_for",
    "num_while",
    "num_switch",
    "o_complexity",
    "max_depth",
    "total_nodes",
    "num_literals",
    "num_ids",
    "unique_ids",
    "id_diversity",
]

In [42]:
def build_decoder_input(emb_df, feat_df):
    # Verificar que ambos dataframes tengan el mismo número de filas
    assert len(emb_df) == len(feat_df), "Los embeddings y las features no tienen el mismo tamaño"

    # Verificar que las etiquetas coincidan
    assert emb_df["label"].reset_index(drop=True).equals(
        feat_df["label"].reset_index(drop=True)
    ), "Las labels no coinciden entre embeddings y features"

    # Embeddings
    X_emb = emb_df.drop("label", axis=1).reset_index(drop=True)

    # Renombrar columnas de embeddings para evitar nombres raros o repetidos
    X_emb.columns = [f"emb_{i}" for i in range(X_emb.shape[1])]

    # Features extra
    X_extra = feat_df[EXTRA_FEATURE_COLUMNS].reset_index(drop=True)

    # Asegurar que todo sea numérico
    X_extra = X_extra.apply(pd.to_numeric, errors="coerce").fillna(0)

    # Combinar embeddings + features estilométricas/AST
    X = pd.concat([X_emb, X_extra], axis=1)

    y = emb_df["label"].reset_index(drop=True)

    return X, y

## 2. Separate attributes and labels


In [47]:
X_train, y_train = build_decoder_input(df_train_emb, df_train_feat)
X_val, y_val = build_decoder_input(df_val_emb, df_val_feat)
X_test, y_test = build_decoder_input(df_test_emb, df_test_feat)

print("\nTrain:", X_train.shape)
print("Train dataframe distribution:")
print(y_train.value_counts().sort_index())

print("\nVal:", X_val.shape)
print("Validation dataframe distribution:")
print(y_val.value_counts().sort_index())

print("\nTest:", X_test.shape)
print("Test dataframe distribution:")
print(y_test.value_counts().sort_index())


Train: (24000, 785)
Train dataframe distribution:
label
0    12000
1    12000
Name: count, dtype: int64

Val: (3000, 785)
Validation dataframe distribution:
label
0    1500
1    1500
Name: count, dtype: int64

Test: (3000, 785)
Test dataframe distribution:
label
0    1500
1    1500
Name: count, dtype: int64


## 4. Import Random Forest model

In [48]:
sys.path.append(os.path.abspath("../models/ML_algorithms"))

from random_forest_model import create_model

rf_model = create_model()

## 5. Train model

In [54]:
y_train_pred = rf_model.predict(X_train)
y_val_pred = rf_model.predict(X_val)
y_test_pred = rf_model.predict(X_test)

rf_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

## 6. Train vs Validation behavior plots (accuracy)

Unlike neural networks, Random Forest models do not learn through epochs or gradient descent. Instead, the model is built by generating multiple decision trees and combining their predictions. Because of this, the behavior is analyzed using the number of trees in the forest rather than epochs.

A traditional loss curve is not included because Random Forest does not optimize a loss function through gradient descent. Instead, each decision tree is constructed by minimizing node impurity using criteria such as Gini Impurity. Therefore, accuracy is a more appropriate metric for analyzing the fitting behavior of the model.

In [55]:
train_acc = []
val_acc = []

for n in range(1, 101):

    rf = RandomForestClassifier(
        n_estimators=n,
        max_depth=10,
        random_state=42
    )

    rf.fit(X_train, y_train)

    train_acc.append(
        accuracy_score(
            y_train,
            rf.predict(X_train)
        )
    )

    val_acc.append(
        accuracy_score(
            y_test,
            rf.predict(X_test)
        )
    )

plt.figure(figsize=(10,5))

plt.plot(
    range(1,101),
    train_acc,
    label="Training Accuracy"
)

plt.plot(
    range(1,101),
    val_acc,
    label="Validation Accuracy"
)

plt.title(
    "Random Forest: Training vs Validation Accuracy"
)

plt.xlabel("Number of Trees")
plt.ylabel("Accuracy")

plt.legend()

plt.show()

KeyboardInterrupt: 

## 7. Predictions

In [ ]:
# # Predictions
# y_train_pred = rf_model.predict(X_train)
# y_test_pred = rf_model.predict(X_test)

## 8. Metrics calculation and classification report.

### Metrics function

In [ ]:
# # Function to evaluate results

# def evaluate_model(y_true, y_pred, dataset_name):
#     accuracy = accuracy_score(y_true, y_pred)

#     precision = precision_score(
#         y_true,
#         y_pred,
#         average="weighted",
#         zero_division=0
#     )

#     recall = recall_score(
#         y_true,
#         y_pred,
#         average="weighted",
#         zero_division=0
#     )

#     f1 = f1_score(
#         y_true,
#         y_pred,
#         average="weighted",
#         zero_division=0
#     )

#     print(f"{dataset_name} Results")
#     print("---------------------")
#     print(f"Accuracy : {accuracy:.4f}")
#     print(f"Precision: {precision:.4f}")
#     print(f"Recall   : {recall:.4f}")
#     print(f"F1 Score : {f1:.4f}")

### Results

In [35]:
# evaluate_model(y_train, y_train_pred, "Training")
# print()
# evaluate_model(y_test, y_test_pred, "Test")

### Classification report

In [36]:
# print(classification_report(y_test, y_test_pred, zero_division=0))

## 9. Confusion matrix

In [37]:
# labels = sorted(y.unique())

# cm = confusion_matrix(
#     y_test,
#     y_test_pred,
#     labels=labels
# )

# plt.figure(figsize=(8, 6))

# sns.heatmap(
#     cm,
#     annot=True,
#     fmt="d",
#     cmap="Blues",
#     xticklabels=labels,
#     yticklabels=labels
# )

# plt.title("Random Forest Confusion Matrix")
# plt.xlabel("Predicted Quality")
# plt.ylabel("Actual Quality")

# plt.show()